# CARA v6 — MPC + SAGA + FINER, user-approved extensions

Default pilot:5 pairs. Set NUM_SAMPLES=200 or1000 later. This version changes the
method; old v6 artifacts cannot be reused because their protocol differs.
Run configuration/setup, self-checks, pair preview, then execute. Models require CUDA,
available checkpoints/Internet and enough memory for Qwen3-VL-8B plus the3 CLIPs.
No pretrained experiment results or VRAM/ASR measurements are bundled.


In [ ]:
NOTEBOOK_CODE_SHA256 = 'c2a2f34a6148e0d3ae42c11a34c2de61ae4bfed8d11618fba7b1839bd91f15f3'

## Provenance and approved changes

- MPC caption/evaluation prompt: `Describe this image in one concise sentence, no longer than 20 words.`
- SAGA attention prompt: `Describe this image, no longer than 25 words.`
- FINER prompts/few-shots/builders/parser and Grad-ECLIP author functions are embedded and hashed.
- MPC source: https://github.com/LiYuanBoJNU/MPCAttack/tree/7b195a5bcbaeaf1e8534d101199810a2782caad3
- SAGA source in repo: `src/SAGA/saga/attention_map.py`, `attention_utils.py`, `text.py`.
- FINER: https://github.com/ExplainableML/finer/tree/d67831696e122d01c9823114835f7f50fa07b44c/train/generate_training_data
- Grad-ECLIP: https://github.com/Cyang-Zhao/Grad-Eclip/blob/e370e6cb194faf2020f5d1ed268f9d57e91a38e6/grad_eclip_image.ipynb

User-approved extensions:K=5 caption banks; all deduplicated FINER objects; one target
bank member nearest the ensemble mean; K source negatives; only3 CLIP encoders;
FINER-matched generated content tokens for SAGA attention; dynamic XAI every30 updates;
one final crop growing from0.5 to1.0, random position from update150.
No custom concept-query instruction is used. No InternVL/DINO features are added.


User-confirmed Grad-ECLIP aggregation: normalize each valid concept map by its sum,
resize to32x32 and renormalize, then take the equal-weight mean. This is a CARA
aggregation choice; the author Grad-ECLIP functions remain unchanged.

In [ ]:
# Configuration — edit this cell, then Run all.
from pathlib import Path
INSTALL_DEPENDENCIES = False
RUN_EXPERIMENT = True
RUN_MODE = "pilot"                 # artifact label: "pilot" | "full"; count uses NUM_SAMPLES
NUM_SAMPLES = 5                     # pilot5; optionally1 for smoke,200 or1000 for full
XAI_METHOD = "grad_eclip"           # "grad_eclip" | "qwen_attention"
DATASET_ROOT = Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset')
SOURCE_DIR = None                    # auto-discover exactly one "bigscale" directory
TARGET_DIR = None                    # auto-discover exactly one "target_images" directory
PAIR_MANIFEST = None                 # JSONL: {"index": 0, "source": "...", "target": "..."}
OUTPUT_ROOT = Path('/kaggle/working/kltn_attack_v6_cara')
SAMPLE_START = 0
SEED = 2026
K = 5
STEPS = 300
REFRESH_EVERY = 30
RANDOM_FROM_STEP = 150               # 1-based: updates 150..300 random, XAI still measured
EPSILON_255 = 16                     # CARA default, not specified by hypothesis
ALPHA_255 = 1                        # MPC pixel-space momentum ascent
TEMPERATURE = 0.2
POSITIVE_OMEGA = 2.0
MOMENTUM_DECAY = 0.9
INPUT_RES = 224
BACKBONES = ('G14', 'B16', 'B32')
CLIP_IDS = {'G14': 'laion/CLIP-ViT-g-14-laion2B-s12B-b42K',
            'B16': 'openai/clip-vit-base-patch16', 'B32': 'openai/clip-vit-base-patch32'}
# Two GPUs: place G14 on GPU1; Qwen+B16+B32 on GPU0. One GPU: leave G14 cuda:0;
# memory is checked by actual execution, with no silent model substitution.
CLIP_DEVICE_MAP = {'G14': 'cuda:1', 'B16': 'cuda:0', 'B32': 'cuda:0'}
ATTACK_DEVICE = 'cuda:0'
QWEN_DEVICE = 'cuda:0'
QWEN_ID = 'Qwen/Qwen3-VL-8B-Instruct'
EVAL_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'  # separate caption VLM, configurable
EVAL_DEVICE = 'cuda:0'
EVALUATE_CAPTIONS = True
MODEL_REVISION_OVERRIDES = {}          # repo_id -> immutable commit; first run resolves once
IMAGE_WEIGHTS = {'G14': 0.6, 'B16': 0.6, 'B32': 0.6}
TEXT_WEIGHTS = {'G14': 0.4, 'B16': 0.4, 'B32': 0.4}
CAPTION_TEMPERATURE = 0.8
CAPTION_TOP_P = 0.9
CAPTION_MAX_NEW_TOKENS = 50            # MPC generate_description
MAX_BANK_ATTEMPTS = 40
QWEN_ATTENTION_LAYER = 29             # SAGA released configuration
QWEN_MAX_PIXELS = None               # use SAGA/processor defaults; attack canvas is224
QWEN_GPU_RESERVE_GIB = 2.0            # execution-only headroom when sharding the8B model
XAI_MAX_SIDE = 448
SAVED_TENSORS_ON_CPU = True           # trade host RAM/PCIe time for attack activation VRAM
LOG_EVERY = 30

# Packages

In [ ]:
# Optional bootstrap. Keep the CUDA-compatible torch/torchvision provided by the host.
if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        'transformers==4.57.1', 'accelerate==1.11.0', 'huggingface_hub>=0.34,<1',
        'ftfy==6.3.1', 'qwen-vl-utils==0.0.14', 'regex', 'Pillow', 'numpy', 'pandas', 'matplotlib', 'nbformat',
        'git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6'], check=True)
    raise SystemExit('Dependencies installed. Restart kernel; set INSTALL_DEPENDENCIES=False.')

# Library

In [ ]:
import ast, contextlib, gc, hashlib, io, json, math, random, re, sys, time
from collections import Counter
from importlib import metadata as package_metadata
from typing import List
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision import datasets
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, InterpolationMode

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def free_cuda():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def image_tensor(image):
    return torch.from_numpy(np.array(image, dtype=np.uint8, copy=True)).permute(2,0,1).unsqueeze(0).float()/255

def tensor_image(x):
    a = x.detach().cpu().squeeze(0).clamp(0,1).mul(255).round().byte().permute(1,2,0).numpy()
    return Image.fromarray(a)

def open_rgb(path):
    with Image.open(path) as im: return ImageOps.exif_transpose(im).convert('RGB')

def reduced_image(image, side=XAI_MAX_SIDE):
    image = image.copy(); image.thumbnail((side, side), Image.Resampling.BICUBIC)
    return image

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def load_attack_image(path):
    # MPC ImageFolder preprocessing: Resize shortest edge, then CenterCrop224.
    transform=Compose([Resize(INPUT_RES,interpolation=InterpolationMode.BICUBIC),
                       T.CenterCrop(INPUT_RES)])
    return transform(open_rgb(path))

## Features, loss and crop

For each backbone, normalize image and text features separately, then fuse
$b_{mj}=0.6u_m+0.4v_{mj}$. Concatenate the3 blocks without per-fused-block normalization.
Choose one actual target member nearest the mean of the K raw concatenated anchors.
The query concatenates normalized image-only CLIP features; global L2 normalization
occurs in the loss, as in MPC.

$$S=-CE([2\cos(q,p)/0.2,\cos(q,n_1)/0.2,\ldots,\cos(q,n_K)/0.2],0).$$

Maximize S with MPC momentum0.9 and sign ascent. All image optimization is on the
MPC Resize+CenterCrop224 canvas in [0,255] units, alpha1 and epsilon16.
The author's initial delta expression `(16/255)*randn` is retained **in these pixel
units**; its standard deviation is about0.0627/255-scale pixels, not16 pixels.
Momentum can keep updating pixels outside the current crop; no extra mask is imposed.

Ten30-update area stages:1/2,5/9,11/18,2/3,13/18,7/9,5/6,8/9,17/18,1.
Steps1–149 use one maximum-mass XAI window; steps150–300 choose random location at
the scheduled size. No nested RandomResizedCrop. The final stage is the full224 canvas.
Maps are measured at0,30,...,300 on the current (8-bit probed) adversarial image.


In [ ]:
def unit(x):
    x = x.float()
    if not torch.isfinite(x).all() or (x.norm(dim=-1) < 1e-12).any():
        raise ValueError('Nonfinite or zero feature vector.')
    return F.normalize(x, dim=-1)

def fuse_features(images, texts, image_weights, text_weights):
    if not len(images) == len(texts) == len(image_weights) == len(text_weights):
        raise ValueError('Backbone fusion lengths differ.')
    blocks = []
    for im, tx, wi, wt in zip(images, texts, image_weights, text_weights):
        if not math.isfinite(wi+wt) or min(wi, wt) < 0 or wi+wt <= 0:
            raise ValueError('Fusion weights must be finite, nonnegative, not both zero.')
        blocks.append(wi*unit(im) + wt*unit(tx))
    return torch.cat(blocks, dim=-1)

def nearest_mean_index(bank):
    if bank.ndim != 2 or len(bank) < 1: raise ValueError('Expected [K,D] bank.')
    return int(((bank-bank.mean(0, keepdim=True))**2).sum(-1).argmin())

def infonce(query, positive, negatives, temperature, omega=2.0):
    if not 0 < temperature < float('inf'): 
        raise ValueError('Invalid temperature.')
    if not 0 < omega < float('inf'): 
        raise ValueError('Invalid positive omega.')
    q, p, n = unit(query), unit(positive), unit(negatives)
    if q.ndim != 2 or p.shape != q.shape or n.ndim != 2 or n.shape[1] != q.shape[1] or len(n) < 1:
        raise ValueError('Expected query/positive [B,D], negatives [K,D].')
    logits = torch.cat([omega*(q*p).sum(-1, keepdim=True), q@n.T], dim=1)/temperature
    return F.cross_entropy(logits, torch.zeros(len(q), dtype=torch.long, device=q.device))

def mpc_infonce_score(query,positive,negatives,temperature=.2,omega=2.0):
    # MPC returns negative CE and performs ascent. K=1 equals author info_nce_loss.
    return -infonce(query,positive,negatives,temperature,omega)

def initialize_mpc_delta(source_cpu,device):
    # Exact MPC expression, in the author's [0,255] optimization space.
    return (16/255*torch.randn_like(source_cpu).to(device)).requires_grad_(True)

def mpc_momentum_step(delta,gradient,momentum,alpha=1.,epsilon=16.,decay=.9):
    if not torch.isfinite(gradient).all(): raise FloatingPointError('Nonfinite MPC gradient.')
    momentum=decay*momentum+gradient
    return (delta.detach()+alpha*momentum.sign()).clamp(-epsilon,epsilon).requires_grad_(True),momentum.detach()

def crop_area(step,steps=300,refresh_every=30):
    if (steps,refresh_every)!=(300,30) or type(step) is not int or not 1<=step<=steps:
        raise ValueError('Expected300 updates,30-update stages and an integer step in[1,300].')
    phase=(step-1)//refresh_every
    return (9+phase)/18  #10 stages:0.5,...,1.0 inclusive

def crop_mode(step, random_from=150):
    return 'random' if step >= random_from else 'xai'

def select_box(heatmap, area, mode, generator):
    heat = torch.as_tensor(heatmap).detach().float().cpu()
    if heat.ndim != 2 or not torch.isfinite(heat).all() or not 0 < area <= 1:
        raise ValueError('Invalid spatial map/area.')
    H, W = heat.shape
    h = max(1, min(H, int(round(H*math.sqrt(area)))))
    w = max(1, min(W, int(round(W*math.sqrt(area)))))
    if area == 1: return (0, 0, H, W)
    if mode == 'random':
        y = int(torch.randint(H-h+1, (1,), generator=generator))
        x = int(torch.randint(W-w+1, (1,), generator=generator))
    elif mode == 'xai':
        # Integral-image exact exhaustive fixed-size window sum, O(HW), no approximate grid.
        sat = F.pad(heat.double(), (1,0,1,0)).cumsum(0).cumsum(1)
        sums = sat[h:,w:]-sat[:-h,w:]-sat[h:,:-w]+sat[:-h,:-w]
        index = int(sums.reshape(-1).argmax()); y, x = divmod(index, W-w+1)
    else: raise ValueError('Unknown crop location mode.')
    return y, x, h, w

def dispersion(heatmap):
    # Same 32x32 normalized coordinates for comparable entropy across patch grids/resolutions.
    heat = torch.as_tensor(heatmap).float().detach().cpu()
    if heat.ndim != 2 or not torch.isfinite(heat).all() or (heat < 0).any() or heat.sum() <= 0:
        raise ValueError('Undefined dispersion: map is nonfinite/negative/zero.')
    p = F.interpolate(heat[None,None], size=(32,32), mode='bilinear', align_corners=False)[0,0]
    p = p/p.sum(); nz = p[p>0]
    entropy = -(nz*nz.log()).sum()
    yy, xx = torch.meshgrid(torch.linspace(0,1,32), torch.linspace(0,1,32), indexing='ij')
    cy, cx = (p*yy).sum(), (p*xx).sum()
    return {'normalized_entropy': float(entropy/math.log(p.numel())),
            'effective_support': float(torch.exp(entropy)/p.numel()),
            'spatial_variance': float((p*((yy-cy)**2+(xx-cx)**2)).sum()),
            'centroid_y': float(cy), 'centroid_x': float(cx)}

In [ ]:
def sample_count():
    if type(NUM_SAMPLES) is not int or NUM_SAMPLES < 1:
        raise ValueError('NUM_SAMPLES must be a positive integer (e.g. 1, 200, 1000).')
    return NUM_SAMPLES

def prepare_pairs():
    n = sample_count()
    def find_dir(label, override):
        if override is not None:
            p = Path(override)
            if not p.is_dir(): raise FileNotFoundError(p)
            return p
        if not DATASET_ROOT.is_dir(): raise FileNotFoundError(f'Edit DATASET_ROOT: {DATASET_ROOT}')
        found = ([DATASET_ROOT] if DATASET_ROOT.name == label else []) + list(DATASET_ROOT.rglob(label))
        found = [p for p in found if p.is_dir()]
        if len(found) != 1: raise ValueError(f'Expected exactly one {label}: {found}. Set explicit directory.')
        return found[0]
    if PAIR_MANIFEST is not None:
        path = Path(PAIR_MANIFEST)
        pairs = [json.loads(s) for s in path.read_text(encoding='utf-8').splitlines() if s.strip()]
        for row in pairs:
            for role in ('source', 'target'):
                p = Path(row[role]); row[role] = str((p if p.is_absolute() else path.parent/p).resolve())
    else:
        # Preserve ImageFolder sorted class/file pairing from the existing attack notebooks.
        s = datasets.ImageFolder(find_dir('bigscale', SOURCE_DIR))
        t = datasets.ImageFolder(find_dir('target_images', TARGET_DIR))
        if len(s) != len(t): raise ValueError('Unequal source/target count; provide an explicit pair manifest.')
        pairs = [{'index': i, 'source': a[0], 'target': b[0]} for i,(a,b) in enumerate(zip(s.samples,t.samples))]
    pairs = pairs[SAMPLE_START:SAMPLE_START+n]
    if len(pairs) != n: raise ValueError(f'Need {n} pairs, found {len(pairs)}.')
    indices = [p['index'] for p in pairs]
    if any(type(i) is not int or i < 0 for i in indices) or len(set(indices)) != n:
        raise ValueError('Pair indices must be unique nonnegative integers.')
    for row in pairs:
        if Path(row['source']).resolve() == Path(row['target']).resolve(): raise ValueError('Identical source/target path.')
        row['input_sha256'] = {r: file_hash(row[r]) for r in ('source','target')}
        if row['input_sha256']['source'] == row['input_sha256']['target']:
            raise ValueError('Identical source and target image bytes.')
    return pairs

def validate_config():
    sample_count()
    if RUN_MODE not in {'pilot','full'} or XAI_METHOD not in {'grad_eclip','qwen_attention'}:
        raise ValueError('Choose a known run mode and XAI method.')
    if (STEPS,REFRESH_EVERY,RANDOM_FROM_STEP) != (300,30,150):
        raise ValueError('v6 fixes 300/30/150. Relabel a schedule ablation explicitly.')
    if type(K) is not int or K < 1 or MAX_BANK_ATTEMPTS < K: raise ValueError('Invalid K/attempt budget.')
    if type(SAMPLE_START) is not int or SAMPLE_START < 0: raise ValueError('Invalid SAMPLE_START.')
    if not 0 < TEMPERATURE < float('inf') or not 0 < ALPHA_255 <= EPSILON_255 <= 255:
        raise ValueError('Invalid loss temperature/pixel budget.')
    if EPSILON_255 != int(EPSILON_255): raise ValueError('Use an integer pixel epsilon for exact PNG bounds.')
    if INPUT_RES!=224 or (QWEN_MAX_PIXELS is not None and QWEN_MAX_PIXELS<32*32) or XAI_MAX_SIDE<32:
        raise ValueError('Expected MPC224 preprocessing and valid XAI limits.')
    if not 0<POSITIVE_OMEGA<float('inf') or not 0<=MOMENTUM_DECAY<1:
        raise ValueError('Invalid MPC omega/momentum.')
    if not 0 < CAPTION_TEMPERATURE < float('inf') or not 0 < CAPTION_TOP_P <= 1:
        raise ValueError('Invalid caption sampling.')
    if EVALUATE_CAPTIONS and EVAL_MODEL_ID == QWEN_ID: raise ValueError('Use a distinct evaluation model.')
    for mapping in (CLIP_IDS,CLIP_DEVICE_MAP,IMAGE_WEIGHTS,TEXT_WEIGHTS):
        if set(mapping) != set(BACKBONES): raise ValueError('Expected exactly G14/B16/B32 keys.')
    for name in BACKBONES:
        wi,wt = IMAGE_WEIGHTS[name],TEXT_WEIGHTS[name]
        if not math.isfinite(wi+wt) or min(wi,wt)<0 or wi+wt<=0: raise ValueError('Invalid fusion weights.')
    if not torch.cuda.is_available(): raise RuntimeError('Pretrained execution requires CUDA.')
    for d in list(CLIP_DEVICE_MAP.values()) + [ATTACK_DEVICE,QWEN_DEVICE,EVAL_DEVICE]:
        device = torch.device(d)
        if device.type != 'cuda' or (device.index or 0) >= torch.cuda.device_count():
            raise RuntimeError(f'{d} unavailable. Edit CLIP_DEVICE_MAP/devices for this host.')
    if package_metadata.version('transformers') != '4.57.1':
        raise RuntimeError('This adapter is pinned to transformers==4.57.1; run optional setup and restart.')

def resolve_revisions():
    from huggingface_hub import HfApi
    lock_path = OUTPUT_ROOT/'model_revisions.json'
    lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
    ids = list(CLIP_IDS.values()) + [QWEN_ID] + ([EVAL_MODEL_ID] if EVALUATE_CAPTIONS else [])
    for model_id in ids:
        rev = MODEL_REVISION_OVERRIDES.get(model_id, lock.get(model_id))
        if rev is None: rev = HfApi().model_info(model_id).sha
        if not re.fullmatch(r'[a-f0-9]{40}', str(rev)): raise ValueError(f'Expected model commit SHA: {model_id}={rev}')
        lock[model_id] = rev
    atomic_json(lock_path, lock)
    return {name: lock[name] for name in sorted(set(ids))}

In [ ]:
# Pinned author definitions. User-approved extensions are outside these strings.
AUTHOR_SOURCES = {'finer_object_prompts.py': 'SYSTEM_MSG_OBJECTS = (\n    "You are an information extraction assistant.\\n"\n    "From the caption, select up to FIVE main objects. A main object must have at least one descriptive attribute in the caption "\n    "(e.g., color, size, material, possession/with-phrase, appositive, relative clause, or an explicit number).\\n"\n    "Rules:\\n"\n    "• Output only object names that appear in the caption and are part of the described scene. Never invent or infer.\\n"\n    "• Prefer plain object names in the output (omit adjectives), but KEEP explicit numbers/quantifiers if the caption states them "\n    "(e.g., \'two cats\', \'five chickens\', \'a pair of skis\').\\n"\n    "• If multiple mentions share the same name and no explicit number is given, output the plural form (e.g., \'dogs\'). \\n"\n    "• List 1–5 main objects; if only one is present, output just that one.\\n"\n    "Do not add quantifiers like \'some\' unless present in the caption.\\n"\n    "Format:\\n"\n    "Return EXACTLY one line:\\n"\n    "PHRASE=<comma-separated list with \'and\' before the last item, e.g., \'a dog, a cat and two birds\'>\\n"\n    "No trailing period. No extra text."\n)\n\nFEW_SHOT_OBJECTS = [\n    (\n        "A bustling street scene with three blue buses at the curb, two red cars in traffic, "\n        "a tall glass office building behind them, and a bronze statue with a raised arm in the plaza.",\n        "PHRASE=three buses, two cars, an office building and a statue"\n    ),\n    (\n        "On a wooden table, a cast-iron skillet sits beside a blue ceramic vase of sunflowers, "\n        "with red apples and green pears piled high in a stainless steel bowl.",\n        "PHRASE=a skillet, a vase, apples, pears and a bowl"\n    ),\n]\n\nSYSTEM_MSG_NEGOBJ = (\n    "You are a negative object creator.\\n"\n    "You will receive a caption, an object list as PHRASE=..., and REPLACE_INDEX=k (1-based).\\n"\n    "Replace EXACTLY the k-th object in PHRASE with a distinctly different NEGATIVE object.\\n"\n    "\\n"\n    "Keep ALL other objects unchanged and preserve their order and punctuation. Keep the same quantifier/determiner "\n    "for the replaced slot (e.g., \'two cats\' -> \'two bicycles\').\\n"\n    "\\n"\n    "Constraints for the NEGATIVE object:\\n"\n    "• It must be distinctly different from the replaced object (not a synonym; not just singular/plural).\\n"\n    "• It must NOT be a synonym or near-equivalent of ANY object that appears in the caption.\\n"\n    "• It must NOT appear anywhere in the caption (as a whole word, singular or plural).\\n"\n    "• Do not modify any other items; do not reorder items; do not add or remove items.\\n"\n    "\\n"\n    "Edge cases (must follow):\\n"\n    "• If REPLACE_INDEX is greater than the number of objects in PHRASE, replace the LAST object.\\n"\n    "• If REPLACE_INDEX is less than 1, replace the FIRST object.\\n"\n    "\\n"\n    "Self-check (must hold):\\n"\n    "• Same number of items as input; exactly one item (the k-th per the rule above) differs.\\n"\n    "\\n"\n    "Output EXACTLY one line:\\n"\n    "PHRASE=<the new list, same format>\\n"\n    "No extra text. No quotes. No trailing period."\n)\n\n# Few-shots cover different list lengths and positions; each uses REPLACE_INDEX to avoid a last-item bias.\nFEW_SHOT_NEGOBJ = [\n    (\n        # Replace FIRST of three\n        "Caption:\\nA small yellow school bus is parked next to two red cars in front of a brick office building.\\n"\n        "PHRASE=a bus, two cars and an office building\\n"\n        "REPLACE_INDEX=1",\n        "PHRASE=a boat, two cars and an office building"\n    ),\n    (\n        "Caption:\\nTwo hikers stand on a wooden bridge with their dog beside them over a calm river.\\n"\n        "PHRASE=two hikers, a dog, a bridge and a river\\n"\n        "REPLACE_INDEX=2",\n        "PHRASE=two hikers, a bicycle, a bridge and a river"\n    ),\n    (\n        "Caption:\\nOn a side table there is a bowl of fruit, a ceramic vase, and a brass lamp.\\n"\n        "PHRASE=a bowl of fruit, a vase and a lamp\\n"\n        "REPLACE_INDEX=3",\n        "PHRASE=a bowl of fruit, a vase and a clock"\n    ),\n    (\n        "Caption:\\nA dashboard card labeled \'Latency\' shows a graph with a menu icon and a legend for several lines.\\n"\n        "PHRASE=a card, a graph, a menu icon, a legend and lines\\n"\n        "REPLACE_INDEX=3",\n        "PHRASE=a card, a graph, a slider, a legend and lines"\n    ),\n    (\n        "Caption:\\nA booking screen shows a highlighted 12:05–12:30 time slot, a search box, an entry for Animal Kingdom Villas, a restaurant icon, and a market link.\\n"\n        "PHRASE=a time slot, a box, an entry, a restaurant and a market\\n"\n        "REPLACE_INDEX=4",\n        "PHRASE=a time slot, a box, an entry, a ticket counter and a market"\n    ),\n]\n\nCATEGORIES = ["natural_image", "chart_graph", "screenshot_ui", "document_text"]\n\nSYSTEM_MSG_CATEGORY = (\n    "You are a strict classifier. Based ONLY on the caption text, assign EXACTLY ONE label.\\n"\n    "Allowed labels:\\n"\n    "  - natural_image\\n"\n    "  - chart_graph   (includes diagrams, flowcharts, schematics, illustrations)\\n"\n    "  - screenshot_ui (UI/app/website/IDE/terminal screenshots)\\n"\n    "  - document_text (scans, printed pages, receipts, invoices, whiteboards, handwriting)\\n"\n    "\\n"\n    "Rules:\\n"\n    "• Return ONLY the label string, nothing else.\\n"\n    "• If multiple seem plausible, pick the single best one.\\n"\n    "• If unsure, choose natural_image.\\n"\n)\n\nFEW_SHOT_CATEGORY = [\n    ("A line chart comparing quarterly revenue for three products with a legend and axes labels.",\n     "chart_graph"),\n    ("A screenshot of a mobile banking app showing the account balance and a transfer button.",\n     "screenshot_ui"),\n    ("A scanned receipt from a grocery store listing items and totals.",\n     "document_text"),\n    ("A photo of a dog running on a beach at sunset.",\n     "natural_image"),\n    ("A flowchart illustrating an authentication pipeline with decision nodes and arrows.",\n     "chart_graph"),\n]\n\n', 'finer_object_helpers.py': 'def _split_objects_phrase(phrase: str) -> List[str]:\n    """\n    Split \'a dog, a cat and two birds\' -> [\'a dog\',\'a cat\',\'two birds\'].\n    Heuristic: split on the last \' and \' and commas on the left.\n    Used only for lightweight validation; storage remains a single string.\n    """\n    s = (phrase or "").strip()\n    if not s:\n        return []\n    if " and " in s:\n        left, last = s.rsplit(" and ", 1)\n        left_parts = [p.strip() for p in left.split(",") if p.strip()]\n        items = left_parts + [last.strip()]\n    else:\n        items = [p.strip() for p in s.split(",") if p.strip()]\n    return items\n\ndef _messages_objects(caption_text: str) -> List[dict]:\n    """Chat messages for the \'objects\' mode with few-shot."""\n    msgs = [{"role": "system", "content": SYSTEM_MSG_OBJECTS}]\n\n    for cap, ans in FEW_SHOT_OBJECTS:\n        demo_user = (\n            "Caption:\\n" + cap + "\\n\\n"\n            "Task: From this caption, select up to five main objects that have at least one descriptive attribute in the caption. "\n            "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "\n            "Return EXACTLY one line:\\nPHRASE=<...>"\n        )\n        msgs.append({"role": "user", "content": demo_user})\n        msgs.append({"role": "assistant", "content": ans})\n\n    task_user = (\n        "Caption:\\n" + caption_text + "\\n\\n"\n        "Task: Select up to five main objects that have at least one descriptive attribute in the caption. "\n        "Prefer plain object names; keep explicit numbers/quantifiers if present; pluralize repeated names without numbers. "\n        "Return EXACTLY one line:\\nPHRASE=<...>"\n    )\n    msgs.append({"role": "user", "content": task_user})\n    return msgs\n\ndef build_objects_prompt(tokenizer, caption: str) -> str:\n    """Final chat prompt using the FIRST 512 caption tokens."""\n    cap_trunc = _truncate_caption_first_tokens(tokenizer, caption, 512)\n    return tokenizer.apply_chat_template(\n        _messages_objects(cap_trunc),\n        tokenize=False,\n        add_generation_prompt=True,\n        enable_thinking=False,\n    )\n\ndef parse_objects_output(text: str) -> str:\n    return parse_tagged_output(\n        text,\n        allowed_tags=("PHRASE",),\n        max_chars=512,\n    )', 'gradeclip_author_functions.py': 'def attention_layer(q, k, v, num_heads=1):\n    "Compute \'Scaled Dot Product Attention\'"\n    tgt_len, bsz, embed_dim = q.shape\n    head_dim = embed_dim // num_heads\n    scaling = float(head_dim) ** -0.5\n    q = q * scaling\n    \n    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)\n    k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)\n    v = v.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)\n    attn_output_weights = torch.bmm(q, k.transpose(1, 2))\n    attn_output_weights = F.softmax(attn_output_weights, dim=-1)\n    attn_output_heads = torch.bmm(attn_output_weights, v)\n    assert list(attn_output_heads.size()) == [bsz * num_heads, tgt_len, head_dim]\n    attn_output = attn_output_heads.transpose(0, 1).contiguous().view(tgt_len, bsz, embed_dim)\n    attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, -1)\n    attn_output_weights = attn_output_weights.sum(dim=1) / num_heads\n    return attn_output, attn_output_weights\n\ndef clip_encode_dense(x,n):\n    vision_width = clipmodel.visual.transformer.width\n    vision_heads = vision_width // 64\n    print("[vision_width and vision_heads]:", vision_width, vision_heads)\n    \n    # modified from CLIP\n    x = x.half()\n    x = clipmodel.visual.conv1(x)  \n    feah, feaw = x.shape[-2:]\n\n    x = x.reshape(x.shape[0], x.shape[1], -1) \n    x = x.permute(0, 2, 1) \n    class_embedding = clipmodel.visual.class_embedding.to(x.dtype)\n    x = torch.cat([class_embedding + torch.zeros(x.shape[0], 1, x.shape[-1]).to(x), x], dim=1)\n\n    ## scale position embedding as the image w-h ratio\n    pos_embedding = clipmodel.visual.positional_embedding.to(x.dtype)\n    tok_pos, img_pos = pos_embedding[:1, :], pos_embedding[1:, :]\n    pos_h = clip_inres // clip_ksize[0]\n    pos_w = clip_inres // clip_ksize[1]\n    assert img_pos.size(0) == (pos_h * pos_w), f"the size of pos_embedding ({img_pos.size(0)}) does not match resolution shape pos_h ({pos_h}) * pos_w ({pos_w})"\n    img_pos = img_pos.reshape(1, pos_h, pos_w, img_pos.shape[1]).permute(0, 3, 1, 2)\n    print("[POS shape]:", img_pos.shape, (feah, feaw))\n    img_pos = torch.nn.functional.interpolate(img_pos, size=(feah, feaw), mode=\'bicubic\', align_corners=False)\n    img_pos = img_pos.reshape(1, img_pos.shape[1], -1).permute(0, 2, 1)\n    pos_embedding = torch.cat((tok_pos[None, ...], img_pos), dim=1)\n    x = x + pos_embedding\n    x = clipmodel.visual.ln_pre(x)\n    \n    x = x.permute(1, 0, 2)  # NLD -> LND\n    x = torch.nn.Sequential(*clipmodel.visual.transformer.resblocks[:-n])(x)\n\n    attns = []\n    atten_outs = []\n    vs = []\n    qs = []\n    ks = []\n    for TR in clipmodel.visual.transformer.resblocks[-n:]:\n        x_in = x\n        x = TR.ln_1(x_in)\n        linear = torch._C._nn.linear    \n        q, k, v = linear(x, TR.attn.in_proj_weight, TR.attn.in_proj_bias).chunk(3, dim=-1)\n        attn_output, attn = attention_layer(q, k, v, 1)  # vision_heads=1\n        attns.append(attn)\n        atten_outs.append(attn_output)\n        vs.append(v)\n        qs.append(q)\n        ks.append(k)\n        \n        x_after_attn = linear(attn_output, TR.attn.out_proj.weight, TR.attn.out_proj.bias)       \n        x = x_after_attn + x_in\n        x = x + TR.mlp(TR.ln_2(x))\n\n    x = x.permute(1, 0, 2)  # LND -> NLD\n    x = clipmodel.visual.ln_post(x)\n    x = x @ clipmodel.visual.proj\n    return x, x_in, vs, qs, ks, attns, atten_outs, (feah, feaw)\n\ndef sim_qk(q, k):\n    q_cls = F.normalize(q[:1,0,:], dim=-1) \n    k_patch = F.normalize(k[1:,0,:], dim=-1)\n\n    cosine_qk = (q_cls * k_patch).sum(-1) \n    cosine_qk_max = cosine_qk.max(dim=-1, keepdim=True)[0]\n    cosine_qk_min = cosine_qk.min(dim=-1, keepdim=True)[0]\n    cosine_qk = (cosine_qk-cosine_qk_min) / (cosine_qk_max-cosine_qk_min)\n    return cosine_qk\n\ndef grad_eclip(c, qs, ks, vs, attn_outputs, map_size):\n    ## gradient on last attention output\n    tmp_maps = []\n    for q, k, v, attn_output in zip(qs, ks, vs, attn_outputs):\n        grad = torch.autograd.grad(\n            c,\n            attn_output,\n            retain_graph=True)[0]\n\n        grad_cls = grad[:1,0,:]\n        v_patch = v[1:,0,:]\n        cosine_qk = sim_qk(q, k).reshape(-1)\n        tmp_maps.append((grad_cls * v_patch * cosine_qk[:,None]).sum(-1)) \n\n    emap = F.relu_(torch.stack(tmp_maps, dim=0)).sum(0)\n    return emap.reshape(*map_size)', 'mpc_info_nce_loss.py': 'def info_nce_loss(feat_adv, feat_tgt, feat_src, temperature=0.2, mode=\'img\', omega=2.0):\n    """\n    InfoNCE loss for pulling adv->tgt close, pushing adv->src away.\n\n    Args:\n        feat_adv: Tensor [B, D] adversarial features\n        feat_tgt: Tensor [B, D] target features (positive)\n        feat_src: Tensor [B, D] source features (negative)\n        temperature: float, softmax temperature\n\n    Returns:\n        loss: scalar tensor\n    """\n    # L2 normalize\n    feat_adv = F.normalize(feat_adv, dim=-1)\n    feat_tgt = F.normalize(feat_tgt, dim=-1)\n    feat_src = F.normalize(feat_src, dim=-1)\n    # Cosine similarities\n    if mode == \'text\':\n        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1).mean(dim=1) * omega / temperature  # [B]\n        sim_neg = torch.sum(feat_adv * feat_src, dim=-1).mean(dim=1) / temperature  # [B]\n    else:\n        sim_pos = torch.sum(feat_adv * feat_tgt, dim=-1) * omega / temperature  # [B]\n        sim_neg = torch.sum(feat_adv * feat_src, dim=-1) / temperature  # [B]\n    # Stack positive and negative: shape [B, 2]\n    logits = torch.stack([sim_pos, sim_neg], dim=1)\n    labels = torch.zeros(feat_adv.size(0), dtype=torch.long, device=feat_adv.device)  # positives index=0\n    # Cross-entropy loss\n    loss = F.cross_entropy(logits, labels)\n    return - loss', 'saga_attention_utils.py': 'def get_token_indices(decoded_text: str, tokenizer, outputs: dict, inputs: dict, vlm_type="qwen3_vl") -> dict:\n    """Extract indices for vision tokens, prompt tokens, and output tokens."""\n    if vlm_type == "qwen3_vl":\n        marker_start = "<|vision_start|>"\n        marker_end = "<|vision_end|>"\n\n    len_prompt_tokens = len(inputs["input_ids"][0])\n    len_all_tokens = len(outputs["sequences"][0])\n\n    # +1 to include <|vision_start|> token\n    vision_token_start = (\n        len(tokenizer(decoded_text.split(marker_start)[0], return_tensors="pt")["input_ids"][0]) + 1\n    )\n    vision_token_end = len(tokenizer(decoded_text.split(marker_end)[0], return_tensors="pt")["input_ids"][0])\n\n    return {\n        "vision_start": vision_token_start,\n        "vision_end": vision_token_end,\n        "output_start": len_prompt_tokens,\n        "output_end": len_all_tokens,\n        "output_len": len_all_tokens - len_prompt_tokens,\n    }\n\ndef aggregate_llm_attention(attn: Tensor, layer_idx: int | None = None) -> Tensor:\n    """\n    Extract average attention vector from attention outputs.\n\n    Nullifies attention over the first token (<bos>), which is a common practice\n    for null attention (https://aclanthology.org/W19-4808.pdf).\n\n    Args:\n        attn: Attention tensor from model outputs\n        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.\n    """\n    if layer_idx is not None and layer_idx != -100:\n        layer = attn[layer_idx]\n        layer_attns = layer.squeeze(0)\n        attns_per_head = layer_attns.mean(dim=0)\n        vec = torch.concat(\n            [\n                torch.tensor([0.0]),  # Zero out null attention on <bos>\n                attns_per_head[-1][1:].cpu(),\n                torch.tensor([0.0]),  # Zero for final generated token\n            ]\n        )\n        return vec / vec.sum()\n\n    avged = []\n    for layer in attn:\n        layer_attns = layer.squeeze(0)\n        attns_per_head = layer_attns.mean(dim=0)\n        vec = torch.concat(\n            [\n                torch.tensor([0.0]),  # Zero out null attention on <bos>\n                attns_per_head[-1][1:].cpu(),\n                torch.tensor([0.0]),  # Zero for final generated token\n            ]\n        )\n        avged.append(vec / vec.sum())\n    return torch.stack(avged).mean(dim=0)\n\ndef heterogenous_stack(vecs: list[torch.Tensor]) -> torch.Tensor:\n    """Pad vectors with zeros to same length then stack."""\n    max_length = max(v.shape[0] for v in vecs)\n    return torch.stack([torch.concat((v, torch.zeros(max_length - v.shape[0]))) for v in vecs])\n\ndef aggregate_prompt_attention(\n    first_generated_token_attn: Tensor,\n    layer_idx: int | None = None,\n) -> Tensor:\n    """\n    Aggregate attention for the prompt/first generated token.\n\n    Args:\n        first_generated_token_attn: Attention tensor from first generation step\n        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.\n    """\n    if layer_idx is not None and layer_idx != -100:\n        layer_attn = first_generated_token_attn[layer_idx]\n        layer_attns = layer_attn.squeeze(0)\n        attns_per_head = layer_attns.mean(dim=0)\n        # last row corresponds to the generated token, exclude it\n        cur = attns_per_head[:-1].cpu().clone()\n\n        # Zero out attention to the first <bos> token (null attention)\n        # Skip first row since <bos> is the only token it can attend to\n        cur[1:, 0] = 0.0\n        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)\n\n        return cur\n\n    aggregated = []\n    for layer_attn in first_generated_token_attn:\n        layer_attns = layer_attn.squeeze(0)\n        attns_per_head = layer_attns.mean(dim=0)\n        # last row corresponds to the generated token, exclude it\n        cur = attns_per_head[:-1].cpu().clone()\n\n        # Zero out attention to the first <bos> token (null attention)\n        # Skip first row since <bos> is the only token it can attend to\n        cur[1:, 0] = 0.0\n        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)\n\n        aggregated.append(cur)\n\n    # Average across all layers\n    return torch.stack(aggregated).mean(dim=0)\n\ndef build_attention_matrix(outputs, layer_idx: int | None = None, valid_token_indices=None) -> torch.Tensor:\n    """\n    Build a complete attention matrix from generation outputs.\n\n    Args:\n        outputs: Model generation outputs with attentions\n        layer_idx: If provided, extract from this layer only. Otherwise average across all layers.\n        valid_token_indices: Optional set of valid token indices to include in the attention matrix.\n\n    Returns:\n        Tensor of shape (total_tokens, max_seq_len) representing attention\n        across all tokens (prompt + generated)\n        entry (i, j): how much token i attends to token j\n    """\n    attention_vectors = []\n\n    # Add <bos> token (attends only to itself)\n    attention_vectors.append(torch.tensor([1.0]))\n\n    # Add prompt token attentions\n    prompt_attention = aggregate_prompt_attention(outputs.attentions[0], layer_idx=layer_idx)\n    attention_vectors.extend(list(prompt_attention))\n\n    # Add generated token attentions\n    for step_idx, step_attention in enumerate(outputs.attentions):\n        if valid_token_indices is not None and step_idx not in valid_token_indices:\n            continue\n        gen_attention = aggregate_llm_attention(step_attention, layer_idx=layer_idx)\n        attention_vectors.append(gen_attention)\n\n    # Pad all vectors to same length and stack\n    return heterogenous_stack(attention_vectors)\n\ndef get_token_img_shape(inputs, processor, vlm_type="qwen3_vl"):\n    if vlm_type == "qwen3_vl":\n        """\n        Qwen3-VL: Calculate the grid dimensions (height, width) of image tokens after spatial merging.\n        NOTE: Processor outputs containing \'image_grid_thw\'.\n        """\n        # Get the raw grid from the processor inputs\n        grid_h = inputs["image_grid_thw"][0][1].item()\n        grid_w = inputs["image_grid_thw"][0][2].item()\n\n        # 2x2 spatial merging/pooling\n        merge_factor = processor.image_processor.merge_size\n        grid_h = grid_h // merge_factor\n        grid_w = grid_w // merge_factor\n        return grid_h, grid_w', 'saga_attention_map.py': 'def get_attention_map_from_generated_text_vlm(\n    model,\n    processor,\n    inputs,\n    vlm_type: str,\n    use_vis_attention: bool = False,\n    temperature: float = 1.0,\n    do_sample: bool = False,\n    layer_idx: int | None = None,\n    use_content_words: bool = False,\n    norm_mode: str = "minmax",\n    **kwargs,\n) -> tuple[Tensor, str]:\n    """Extract generation-time Qwen3-VL attention using the original causal row alignment."""\n\n    if vlm_type != "qwen3_vl":\n        raise ValueError("This release supports Qwen3-VL only.")\n\n    tokenizer = processor.tokenizer\n\n    outputs = model.generate(\n        **inputs,\n        max_new_tokens=50,\n        return_dict_in_generate=True,\n        output_attentions=True,\n        do_sample=do_sample,\n        temperature=temperature if do_sample else 1.0,\n    )\n\n    decoded_text = processor.batch_decode(\n        outputs["sequences"], skip_special_tokens=False, clean_up_tokenization_spaces=False\n    )[0]\n\n    token_indices = get_token_indices(decoded_text, tokenizer, outputs, inputs, vlm_type=vlm_type)\n    generated_token_ids = outputs.sequences[0][token_indices["output_start"] : token_indices["output_end"]]\n    valid_token_indices = filter_generated_tokens(tokenizer, generated_token_ids, use_content_words)\n\n    # Build attention matrix and get token indices\n    # Keep every generation step: compacting filtered rows breaks causal row alignment.\n    llm_attn_matrix = build_attention_matrix(outputs=outputs, layer_idx=layer_idx)\n\n    vision_token_start = token_indices["vision_start"]\n    vision_token_end = token_indices["vision_end"]\n    num_vision_tokens = vision_token_end - vision_token_start\n\n    grid_h, grid_w = get_token_img_shape(inputs=inputs, processor=processor, vlm_type=vlm_type)\n    assert grid_h * grid_w == num_vision_tokens, (\n        f"Mismatch in vision tokens ({num_vision_tokens}) and grid size: {num_vision_tokens} vs {grid_h}x{grid_w}"\n    )\n\n    # Average attention maps over all valid generated tokens\n    attn_maps = []\n    for token_offset in valid_token_indices:\n        token_row = token_indices["output_start"] + token_offset\n        if token_row >= llm_attn_matrix.shape[0]:\n            logger.warning(\n                f"Token row {token_row} out of range for attention matrix with shape {llm_attn_matrix.shape}"\n            )\n            continue\n\n        attn_weights_over_vis_tokens = llm_attn_matrix[token_row][vision_token_start:vision_token_end]\n        if attn_weights_over_vis_tokens.sum() == 0:\n            continue\n        attn_weights_over_vis_tokens = attn_weights_over_vis_tokens / attn_weights_over_vis_tokens.sum()\n        attn_maps.append(attn_weights_over_vis_tokens.reshape(grid_h, grid_w))\n\n    if attn_maps:\n        attn_over_image = torch.stack(attn_maps).mean(dim=0)\n    else:\n        attn_over_image = torch.zeros((grid_h, grid_w), dtype=llm_attn_matrix.dtype)\n\n    mn = attn_over_image.min()\n    mx = attn_over_image.max()\n    attn_over_image = (attn_over_image - mn) / (mx - mn + 1e-8)\n    logger.debug(f"Attention map shape: {attn_over_image.shape}")\n\n    generated_text = tokenizer.decode(\n        outputs.sequences[0][token_indices["output_start"] : token_indices["output_end"]],\n        skip_special_tokens=True,\n    )\n\n    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True)\n    return attn_over_image, generated_text', 'saga_text.py': 'STOP_WORDS = {\n    "i",\n    "me",\n    "my",\n    "myself",\n    "we",\n    "our",\n    "ours",\n    "ourselves",\n    "you",\n    "you\'re",\n    "you\'ve",\n    "you\'ll",\n    "you\'d",\n    "your",\n    "yours",\n    "yourself",\n    "yourselves",\n    "he",\n    "him",\n    "his",\n    "himself",\n    "she",\n    "she\'s",\n    "her",\n    "hers",\n    "herself",\n    "it",\n    "it\'s",\n    "its",\n    "itself",\n    "they",\n    "them",\n    "their",\n    "theirs",\n    "themselves",\n    "what",\n    "which",\n    "who",\n    "whom",\n    "this",\n    "that",\n    "that\'ll",\n    "these",\n    "those",\n    "am",\n    "is",\n    "are",\n    "was",\n    "were",\n    "be",\n    "been",\n    "being",\n    "have",\n    "has",\n    "had",\n    "having",\n    "do",\n    "does",\n    "did",\n    "doing",\n    "a",\n    "an",\n    "the",\n    "and",\n    "but",\n    "if",\n    "or",\n    "because",\n    "as",\n    "until",\n    "while",\n    "of",\n    "at",\n    "by",\n    "for",\n    "with",\n    "about",\n    "against",\n    "between",\n    "into",\n    "through",\n    "during",\n    "before",\n    "after",\n    "above",\n    "below",\n    "to",\n    "from",\n    "up",\n    "down",\n    "in",\n    "out",\n    "on",\n    "off",\n    "over",\n    "under",\n    "again",\n    "further",\n    "then",\n    "once",\n    "here",\n    "there",\n    "when",\n    "where",\n    "why",\n    "how",\n    "all",\n    "any",\n    "both",\n    "each",\n    "few",\n    "more",\n    "most",\n    "other",\n    "some",\n    "such",\n    "no",\n    "nor",\n    "not",\n    "only",\n    "own",\n    "same",\n    "so",\n    "than",\n    "too",\n    "very",\n    "s",\n    "t",\n    "can",\n    "will",\n    "just",\n    "don",\n    "don\'t",\n    "should",\n    "should\'ve",\n    "now",\n    "d",\n    "ll",\n    "m",\n    "o",\n    "re",\n    "ve",\n    "y",\n    "ain",\n    "aren",\n    "aren\'t",\n    "couldn",\n    "couldn\'t",\n    "didn",\n    "didn\'t",\n    "doesn",\n    "doesn\'t",\n    "hadn",\n    "hadn\'t",\n    "hasn",\n    "hasn\'t",\n    "haven",\n    "haven\'t",\n    "isn",\n    "isn\'t",\n    "ma",\n    "mightn",\n    "mightn\'t",\n    "mustn",\n    "mustn\'t",\n    "needn",\n    "needn\'t",\n    "shan",\n    "shan\'t",\n    "shouldn",\n    "shouldn\'t",\n    "wasn",\n    "wasn\'t",\n    "weren",\n    "weren\'t",\n    "won",\n    "won\'t",\n    "wouldn",\n    "wouldn\'t",\n}\n\ndef clean_token(token: str) -> str:\n    """Lowercase, strip, and drop punctuation to normalize tokens."""\n    return token.strip().lower().translate(str.maketrans("", "", string.punctuation))\n\ndef is_content_word(token: str) -> bool:\n    """Return True if token is non-empty after cleaning and not a stop word."""\n    cleaned = clean_token(token)\n    return bool(cleaned) and cleaned not in STOP_WORDS\n\ndef filter_generated_tokens(tokenizer, token_ids: Sequence[int], use_content_words: bool) -> list[int]:\n    """\n    Return indices of tokens to use. If use_content_words is True, keep only content\n    words; otherwise return all indices. Falls back to all tokens if filtering empties.\n    """\n    if not use_content_words:\n        return list(range(len(token_ids)))\n\n    logger.debug(f"[Use content words: {use_content_words}] Only keeping content words")\n    valid_indices = []\n    for idx, token_id in enumerate(token_ids):\n        token_str = tokenizer.decode([token_id], skip_special_tokens=False)\n        if is_content_word(token_str):\n            valid_indices.append(idx)\n        elif DEBUG:\n            logger.debug(f"Filtering out token \'{token_str}\' at index {idx}")\n\n    if not valid_indices:\n        return list(range(len(token_ids)))\n\n    return valid_indices', 'finer_general_helpers.py': 'def truncate_first_tokens(tokenizer, text: str, max_tokens: int = 512) -> str:\n    """\n    Keep only the first `max_tokens` tokenizer tokens.\n    """\n    ids = tokenizer(text or "", add_special_tokens=False).input_ids\n    ids = ids[:max_tokens]\n    return tokenizer.decode(ids, skip_special_tokens=True).strip()\n\ndef parse_tagged_output(\n    text: str,\n    *,\n    allowed_tags: Sequence[str],\n    max_chars: int = 512,\n    split_on_semicolon: bool = False,\n    strip_trailing_pattern: str = r"[.;:\\s]+$",\n) -> str:\n    """\n    Parse a generated single-line output like:\n      PHRASE=...\n      SENTENCE=...\n      ACC_PHRASE=...\n    by scanning for the first matching tagged line.\n\n    Fallback:\n      - if no tagged line is found, use the first non-empty line.\n\n    Normalization:\n      - strip outer quotes\n      - optional split on first \';\'\n      - strip trailing punctuation via regex\n      - collapse internal whitespace\n      - clip to max_chars\n    """\n    raw = (text or "").strip()\n    lines = [\n        line.strip()\n        for line in raw.splitlines()\n        if line.strip() and line.strip() not in {"```", "~~~"}\n    ]\n\n    if not lines:\n        return ""\n\n    tag_pattern = "|".join(re.escape(tag) for tag in allowed_tags)\n    pattern = re.compile(\n        rf"^\\s*(?:{tag_pattern})\\s*[:=]\\s*(.+)$",\n        flags=re.IGNORECASE,\n    )\n\n    value = ""\n    for line in lines:\n        match = pattern.match(line)\n        if match:\n            value = match.group(1).strip()\n            break\n\n    if not value:\n        value = lines[0].strip()\n\n    value = (\n        value.strip()\n        .strip(\'"\')\n        .strip("\'")\n        .strip("“”‘’")\n        .strip()\n    )\n\n    if split_on_semicolon:\n        value = value.split(";", 1)[0].strip()\n\n    if strip_trailing_pattern:\n        value = re.sub(strip_trailing_pattern, "", value)\n\n    value = re.sub(r"\\s+", " ", value).strip()\n\n    if max_chars is not None:\n        value = value[:max_chars]\n\n    return value'}
AUTHOR_SHA256 = {'finer_object_prompts.py': '4834a17db74b9a716865954a98b78fd3dafcaebc2434f1ad78fd2a8336d9fd77', 'finer_object_helpers.py': '97fe84f6b421ef8e8c7a3cbe1aa3551911d50ac6931d7165a8b8da9bb520e9be', 'gradeclip_author_functions.py': '41d72e9c60c771955be9460b50132c3bfe928e68033e68cbe3bc8b01bcec6838', 'mpc_info_nce_loss.py': '857c91ff8e6fe6290a3e569135b97cc2c2898c55a22347a294a184e068ee804c', 'saga_attention_utils.py': 'af68ce21bded5d34e138c7059f313daa34d5a8d055082a940cb0e34a011f4983', 'saga_attention_map.py': 'e408b4168e02c2f2ef0e0d41f9284295a66e546b7067a80d049b2d947711dee7', 'saga_text.py': '8205a79ed5fe0d1588091e6520a00ef9941fa3441f691cd0fcb92c5a26b50abc', 'finer_general_helpers.py': '5f8970a2ee394b75eb8640ccea43bb69764ab6b3f858cbed903f0f9f7cd3b5e4'}
AUTHOR_REVISIONS = {'FINER':'d67831696e122d01c9823114835f7f50fa07b44c',
    'MPC':'7b195a5bcbaeaf1e8534d101199810a2782caad3',
    'SAGA':'042274599b958d4d63760b9bf44bbcb88493811c',
    'Grad-ECLIP':'e370e6cb194faf2020f5d1ed268f9d57e91a38e6',
    'OpenAI-CLIP':'d05afc436d78f1c48dc0dbf8e5980a9d471f35f6','Transformers':'v4.57.1'}

def verify_author_sources():
    for name,src in AUTHOR_SOURCES.items():
        if hashlib.sha256(src.encode()).hexdigest()!=AUTHOR_SHA256[name]:
            raise RuntimeError(f'Author source modified: {name}')
        ast.parse(src)

verify_author_sources()
from collections.abc import Sequence
import logging,string
FINER_NS={'List':List,'Sequence':Sequence,'re':re}
exec(compile(AUTHOR_SOURCES['finer_object_prompts.py'],'<finer-prompts>','exec'),FINER_NS)
exec(compile(AUTHOR_SOURCES['finer_general_helpers.py'],'<finer-general>','exec'),FINER_NS)
FINER_NS['_truncate_caption_first_tokens']=FINER_NS['truncate_first_tokens']
exec(compile(AUTHOR_SOURCES['finer_object_helpers.py'],'<finer-objects>','exec'),FINER_NS)
MPC_NS={'torch':torch,'F':F}
exec(compile(AUTHOR_SOURCES['mpc_info_nce_loss.py'],'<mpc-loss>','exec'),MPC_NS)
SAGA_NS={'torch':torch,'Tensor':torch.Tensor,'Sequence':Sequence,'string':string,
          'logger':logging.getLogger('cara.saga'),'DEBUG':False}
for name in ('saga_attention_utils.py','saga_text.py','saga_attention_map.py'):
    exec(compile(AUTHOR_SOURCES[name],'<'+name+'>','exec'),SAGA_NS)

In [ ]:
# Instruction prompts copied verbatim from released MPC/SAGA code.
CAPTION_PROMPT = 'Describe this image in one concise sentence, no longer than 20 words.'
EVAL_CAPTION_PROMPT = CAPTION_PROMPT
ATTENTION_PROMPT = 'Describe this image, no longer than 25 words.'

class QwenCaptioner:
    def __init__(self, model_id, revision, device, attention=False):
        from transformers import AutoProcessor, AutoModelForImageTextToText
        self.device = torch.device(device)
        processor_options={} if QWEN_MAX_PIXELS is None else {'max_pixels':QWEN_MAX_PIXELS}
        self.processor=AutoProcessor.from_pretrained(model_id,revision=revision,**processor_options)
        with torch.cuda.device(self.device):
            dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        # Place the8B model after accounting for resident CLIPs; no quantization/substitution.
        max_memory={}
        for i in range(torch.cuda.device_count()):
            free,_=torch.cuda.mem_get_info(i)
            max_memory[i]=max(1,int(free-QWEN_GPU_RESERVE_GIB*2**30))
        self.model=AutoModelForImageTextToText.from_pretrained(model_id,revision=revision,
            torch_dtype=dtype,attn_implementation='eager' if attention else 'sdpa',
            device_map='auto',max_memory=max_memory,low_cpu_mem_usage=True)
        self.device=self.model.device
        self.model.eval().requires_grad_(False)

    def batch(self, messages, image=None):
        prompt = messages if isinstance(messages,str) else self.processor.apply_chat_template(
            messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
        kwargs = {'text': [prompt], 'return_tensors': 'pt'}
        if image is not None: kwargs['images'] = [image]
        return self.processor(**kwargs).to(self.device)

    @torch.inference_mode()
    def generate(self, messages, image=None, sample=False, max_new_tokens=96):
        batch = self.batch(messages,image)
        opts = dict(max_new_tokens=max_new_tokens, do_sample=sample, use_cache=True)
        if sample: opts.update(temperature=CAPTION_TEMPERATURE, top_p=CAPTION_TOP_P)
        ids = self.model.generate(**batch, **opts)
        result = self.processor.batch_decode(ids[:,batch['input_ids'].shape[1]:],
                      skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        if not result: raise ValueError('VLM returned an empty response.')
        return result

    def caption(self, image, sample=False, evaluation=False):
        messages = [{'role':'user','content':[{'type':'image'},
            {'type':'text','text': EVAL_CAPTION_PROMPT if evaluation else CAPTION_PROMPT}]}]
        return self.generate(messages,image,sample,CAPTION_MAX_NEW_TOKENS)

    def object_record(self,caption):
        # Original FINER512-token truncation, prompt builder and parser.
        prompt=FINER_NS['build_objects_prompt'](self.processor.tokenizer,caption)
        raw=self.generate(prompt,max_new_tokens=64)  # FINER build_generate_obj_parser default
        phrase=FINER_NS['parse_objects_output'](raw)
        objects=FINER_NS['_split_objects_phrase'](phrase)
        if not 1<=len(objects)<=5: raise ValueError(f'FINER produced no valid object list: {raw!r}')
        return {'raw_response':raw,'phrase':phrase,'objects':objects}

    def saga_batch(self,image):
        # Same image serialization/vision preparation as SAGA attention.py.
        import base64
        from qwen_vl_utils import process_vision_info
        image_processor=self.processor.image_processor
        if getattr(image_processor,'min_pixels',None) is None:image_processor.min_pixels=200704
        if getattr(image_processor,'max_pixels',None) is None:image_processor.max_pixels=1003520
        buffer=io.BytesIO();image.save(buffer,format='PNG')
        messages=[{'role':'user','content':[
            {'type':'image','image':'data:image/png;base64,'+base64.b64encode(buffer.getvalue()).decode()},
            {'type':'text','text':ATTENTION_PROMPT}]}]
        images,videos,_=process_vision_info(messages,image_patch_size=16,
            return_video_kwargs=True,return_video_metadata=True)
        text=self.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        return self.processor(text=text,images=images,videos=videos,do_resize=False,
                              return_tensors='pt').to(self.device)


def caption_bank(captioner, image):
    captions, attempts = [], []
    for _ in range(MAX_BANK_ATTEMPTS):
        raw = captioner.caption(image, sample=True)
        cleaned = ' '.join(raw.split()); key = cleaned.casefold().rstrip('. ')
        duplicate = key in {c.casefold().rstrip('. ') for c in captions}
        attempts.append({'raw_response':raw,'accepted':not duplicate})
        if not duplicate: captions.append(cleaned)
        if len(captions)==K: break
    if len(captions)!=K:
        raise RuntimeError(f'Only {len(captions)}/{K} unique captions; increase attempt budget. No duplicate padding.')
    objects = [captioner.object_record(c) for c in captions]
    selected=[];seen=set()
    for row in objects:
        for phrase in row['objects']:
            key=phrase.casefold()
            if key not in seen: selected.append(phrase);seen.add(key)
    if not selected: raise ValueError('FINER bank has no objects.')
    return {'captions':captions,'caption_attempts':attempts,'object_records':objects,
            'selected_concepts':selected}

In [ ]:
class ClipEnsemble:
    def __init__(self, revisions):
        from transformers import CLIPModel, AutoTokenizer
        self.device = torch.device(ATTACK_DEVICE)
        self.models, self.tokenizers = {}, {}
        for name in BACKBONES:
            ident = CLIP_IDS[name]
            model = CLIPModel.from_pretrained(ident, revision=revisions[ident], torch_dtype=torch.float32)
            model.eval().requires_grad_(False).to(CLIP_DEVICE_MAP[name])
            self.models[name] = model
            self.tokenizers[name] = AutoTokenizer.from_pretrained(ident, revision=revisions[ident])

    def images(self, image):
        features = []
        for name in BACKBONES:
            model = self.models[name]; d = next(model.parameters()).device
            x = F.interpolate(image.to(d), size=(224,224), mode='bicubic', align_corners=False,
                              antialias=True).clamp(0,1)
            mean = x.new_tensor([.48145466,.4578275,.40821073])[None,:,None,None]
            std = x.new_tensor([.26862954,.26130258,.27577711])[None,:,None,None]
            features.append(unit(model.get_image_features(pixel_values=(x-mean)/std)).to(self.device))
        return features

    @torch.no_grad()
    def texts(self, texts):
        features = []
        for name in BACKBONES:
            model = self.models[name]; tokenizer = self.tokenizers[name]
            tokens = tokenizer(texts, padding=True, truncation=False, return_tensors='pt')
            if tokens.input_ids.shape[1] > model.config.text_config.max_position_embeddings:
                raise ValueError(f'{name}: caption exceeds CLIP token limit; shorten generation prompt/token budget.')
            tokens = tokens.to(next(model.parameters()).device)
            features.append(unit(model.get_text_features(**tokens)).to(self.device))
        return features

    def query(self, image):
        return torch.cat(self.images(image),dim=-1)

    @torch.no_grad()
    def anchor_bank(self, image, captions):
        return fuse_features(self.images(image), self.texts(captions),
            [IMAGE_WEIGHTS[n] for n in BACKBONES], [TEXT_WEIGHTS[n] for n in BACKBONES])

    @torch.no_grad()
    def anchors(self, source, target, banks):
        target_bank = self.anchor_bank(target,banks['target']['captions'])
        source_bank = self.anchor_bank(source,banks['source']['captions'])
        j = nearest_mean_index(target_bank)
        return target_bank[j:j+1].detach(),source_bank.detach(),{
            'target_index':j, 'target_caption':banks['target']['captions'][j],
            'target_distance_to_mean':((target_bank-target_bank.mean(0))**2).sum(-1).sqrt().cpu().tolist(),
            'source_pairwise_cosine':(unit(source_bank)@unit(source_bank).T).cpu().tolist(),
            'target_bank':target_bank.cpu().tolist(),'source_bank':source_bank.cpu().tolist()}

In [ ]:
class GradEclipXAI:
    def __init__(self, device):
        import clip
        self.clip = clip
        # Original OpenAI CLIP FP16 GPU model; separate from the HF ensemble.
        model, _ = clip.load('ViT-B/16', device=device, jit=False)
        model.eval().requires_grad_(False)
        self.model, self.device = model,torch.device(device)
        self.ns = dict(torch=torch,F=F,clipmodel=model,
            clip_inres=int(model.visual.input_resolution),clip_ksize=model.visual.conv1.kernel_size)
        exec(compile(AUTHOR_SOURCES['gradeclip_author_functions.py'],'<gradeclip-author>','exec'),self.ns)

    def maps(self, image, concepts):
        # Author imgprocess recipe: resize to nearest patch multiple while preserving aspect.
        image = reduced_image(image)
        w,h = image.size; ph,pw = self.ns['clip_ksize']
        nw,nh = max(pw,int(w/pw+.5)*pw),max(ph,int(h/ph+.5)*ph)
        preprocess = Compose([Resize((nh,nw), interpolation=InterpolationMode.BICUBIC),ToTensor(),
            Normalize((.48145466,.4578275,.40821073),(.26862954,.26130258,.27577711))])
        x = preprocess(image).unsqueeze(0).to(self.device).requires_grad_(True)
        with torch.no_grad():
            text = unit(self.model.encode_text(self.clip.tokenize(concepts,truncate=False).to(self.device)))
        with torch.enable_grad(), contextlib.redirect_stdout(io.StringIO()):
            outputs,_,vs,qs,ks,_,attention_outputs,grid = self.ns['clip_encode_dense'](x,n=1)
            scores = (unit(outputs[:,0])@text.T)[0]
            maps = [self.ns['grad_eclip'](s,qs,ks,vs,attention_outputs,grid).detach().float().cpu()
                    for s in scores]
        return maps,{'grid':list(grid),'model':'OpenAI ViT-B/16','n':1,
                     'author_forward_cosine':scores.detach().float().cpu().tolist()}

class ConceptMatchError(ValueError):
    def __init__(self,message,details):
        super().__init__(message);self.details=details

def match_finer_generated_tokens(tokenizer,token_ids,concepts,content_offsets):
    ids=[int(x) for x in token_ids]
    text=tokenizer.decode(ids,skip_special_tokens=False,clean_up_tokenization_spaces=False)
    # Prefix decoding preserves real BPE spans; do not match 'car' inside 'carpet'.
    prefixes=[tokenizer.decode(ids[:i],skip_special_tokens=False,clean_up_tokenization_spaces=False)
              for i in range(len(ids)+1)]
    if any(not text.startswith(p) for p in prefixes) or any(len(a)>len(b) for a,b in zip(prefixes,prefixes[1:])):
        raise ConceptMatchError('Non-monotonic generated-token decoding; cannot align concepts.',
                                {'generated_text':text,'concepts':concepts})
    spans=[(len(prefixes[i]),len(prefixes[i+1])) for i in range(len(ids))]
    allowed=set(content_offsets);selected=set();matches=[]
    # Approved rule: content words normalized by SAGA; no article/phrase exactness,
    # synonyms or stemming. Match complete decoded words before mapping back to BPEs.
    word_pattern=r"\w+(?:['’-]\w+)*"
    concept_words={phrase:{SAGA_NS['clean_token'](m.group()) for m in re.finditer(word_pattern,phrase)
                          if SAGA_NS['is_content_word'](m.group())} for phrase in concepts}
    for phrase in concepts:
        occurrences=[(m.start(),m.end()) for m in re.finditer(word_pattern,text)
                     if SAGA_NS['clean_token'](m.group()) in concept_words[phrase]]
        offsets=sorted({i for i,(a,b) in enumerate(spans) if i in allowed
                        and any(a<end and b>start for start,end in occurrences)})
        if offsets:
            selected.update(offsets);matches.append({'concept':phrase,'char_spans':occurrences,'token_offsets':offsets})
    details={'generated_text':text,'concepts':concepts,'matched_concepts':[r['concept'] for r in matches],
             'matches':matches,'selected_token_offsets':sorted(selected),
             'generated_token_ids':ids,'concept_content_words':{p:sorted(w) for p,w in concept_words.items()},
             'matching':'whole content words using SAGA normalization; no synonyms/lemmatization'}
    if not selected: raise ConceptMatchError('No generated content tokens match the FINER concepts.',details)
    return details

class QwenAttentionXAI:
    def __init__(self,captioner):self.captioner=captioner

    @torch.no_grad()
    def maps(self,image,concepts):
        cap=self.captioner;batch=cap.saga_batch(image);ns=dict(SAGA_NS)
        original_filter=SAGA_NS['filter_generated_tokens'];selection={}
        def finer_filter(tokenizer,token_ids,use_content_words):
            allowed=original_filter(tokenizer,token_ids,use_content_words)
            selection.update(match_finer_generated_tokens(tokenizer,token_ids,concepts,allowed))
            return selection['selected_token_offsets']
        ns['filter_generated_tokens']=finer_filter
        # Recompile only the exact author function to give it this call's private namespace.
        exec(compile(AUTHOR_SOURCES['saga_attention_map.py'],'<saga-finer-extension>','exec'),ns)
        attention,caption=ns['get_attention_map_from_generated_text_vlm'](
            cap.model,cap.processor,batch,vlm_type='qwen3_vl',layer_idx=QWEN_ATTENTION_LAYER,
            use_content_words=True,do_sample=False,norm_mode='minmax')
        # Same second min-max normalization / constant-map handling as SAGA attention.py.
        array=attention.detach().float().cpu().numpy();mn,mx=array.min(),array.max()
        array=(array-mn)/(mx-mn) if mx>mn else np.ones_like(array)*.5
        if not np.isfinite(array).all():raise FloatingPointError('SAGA returned a nonfinite attention map.')
        return [torch.from_numpy(array.copy())],{'map_labels':['saga_finer_generated'],
            'layer':QWEN_ATTENTION_LAYER,'model':QWEN_ID,'generated_caption':caption,
            'selection':selection,'aggregation':'SAGA per-generated-token image normalization, mean, minmax'}

In [ ]:
def refresh_xai(xai, adversarial, concepts, step, folder):
    try:maps,meta=xai.maps(tensor_image(adversarial),concepts)
    except ConceptMatchError as error:
        atomic_json(Path(folder)/f'xai_failure_{step:03d}.json',
                    {'step':step,'error':str(error),**error.details})
        raise
    valid=[]; errors=[]; arrays={}
    labels=meta.get('map_labels',concepts)
    if len(labels)!=len(maps):raise ValueError('XAI map/label count mismatch.')
    for i,(phrase,m) in enumerate(zip(labels,maps)):
        arrays[f'concept_{i}']=m.numpy()
        if not torch.isfinite(m).all() or (m<0).any() or m.sum()<=0:
            errors.append({'concept':phrase,'reason':'nonfinite/negative/zero author map'})
            continue
        # Equal weight per valid concept, preserving spatial distribution within each map.
        m=m/m.sum()
        valid.append(F.interpolate(m[None,None],size=(32,32),mode='bilinear',align_corners=False)[0,0])
    if not valid: raise RuntimeError(f'No valid {XAI_METHOD} concept maps at step {step}: {errors}')
    mean=torch.stack([m/m.sum() for m in valid]).mean(0)
    H,W=adversarial.shape[-2:]
    full=F.interpolate(mean[None,None],size=(H,W),mode='bilinear',align_corners=False)[0,0]
    record={'step':step,'method':XAI_METHOD,'concepts':concepts,'excluded_maps':errors,
            'dispersion':dispersion(mean),'metadata':meta}
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    arrays['aggregate_32']=mean.numpy()
    np.savez_compressed(folder/f'xai_{step:03d}.npz',**arrays)
    atomic_json(folder/f'xai_{step:03d}.json',record)
    return full,record

def attack_one(ensemble,xai,source,target,banks,folder,pair_seed):
    seed_everything(pair_seed)
    source255=(source.to(ATTACK_DEVICE)*255).round()
    delta=initialize_mpc_delta(source,ATTACK_DEVICE)
    momentum=torch.zeros_like(delta,requires_grad=False)
    positive,negatives,anchor_record=ensemble.anchors(source255/255,target.to(ATTACK_DEVICE),banks)
    atomic_json(folder/'anchors.json',anchor_record)
    concepts=banks['source']['selected_concepts']
    generator=torch.Generator(device='cpu').manual_seed(pair_seed)
    # Step0 measures the initial adversarial state, including MPC's random start.
    heat,record=refresh_xai(xai,((source255+delta)/255).clamp(0,1),concepts,0,folder/'xai')
    xai_log=[record];history=[];cached_box=None
    for step in range(1,STEPS+1):
        area=crop_area(step,STEPS,REFRESH_EVERY);mode=crop_mode(step,RANDOM_FROM_STEP)
        if mode=='xai':
            if cached_box is None:cached_box=select_box(heat,area,mode,generator)
            box=cached_box
        else:box=select_box(heat,area,mode,generator)
        y,x,h,w=box
        context=torch.autograd.graph.save_on_cpu(pin_memory=True) if SAVED_TENSORS_ON_CPU else contextlib.nullcontext()
        with context:
            query=ensemble.query((source255+delta)[:,:,y:y+h,x:x+w]/255)
            score=mpc_infonce_score(query,positive,negatives,TEMPERATURE,POSITIVE_OMEGA)
            if not torch.isfinite(score):raise FloatingPointError(f'Nonfinite MPC score at{step}.')
            gradient,=torch.autograd.grad(score,delta)
        grad_l1=float(gradient.abs().sum())
        if grad_l1==0:raise RuntimeError(f'Zero gradient at update{step}.')
        delta,momentum=mpc_momentum_step(delta,gradient,momentum,ALPHA_255,EPSILON_255,MOMENTUM_DECAY)
        history.append({'step':step,'loss_before_update':float(-score.detach()),
            'mpc_score_before_update':float(score.detach()),'box_yxhw':list(box),'scheduled_area':area,
            'actual_area':h*w/(source.shape[-2]*source.shape[-1]),'crop_mode':mode,'gradient_l1':grad_l1})
        del query,score,gradient
        if step%REFRESH_EVERY==0:
            heat,record=refresh_xai(xai,((source255+delta)/255).clamp(0,1),concepts,step,folder/'xai')
            xai_log.append(record);cached_box=None
            atomic_json(folder/'progress.json',{'completed_updates':step,'history':history,'xai':xai_log})
        if step%LOG_EVERY==0:print(f'  update{step}/{STEPS}, {mode}, area={area:.4f}, CE={history[-1]["loss_before_update"]:.4f}')
    adversarial=((source255+delta)/255).clamp(0,1).detach()
    with torch.no_grad():
        qs,qa=unit(ensemble.query(source255/255)),unit(ensemble.query(adversarial))
        p,n=unit(positive),unit(negatives)
        scores={'clean_global_infonce':float(infonce(qs,positive,negatives,TEMPERATURE,POSITIVE_OMEGA)),
            'adversarial_global_infonce':float(infonce(qa,positive,negatives,TEMPERATURE,POSITIVE_OMEGA)),
            'clean_target_cosine':float((qs*p).sum()),'adversarial_target_cosine':float((qa*p).sum()),
            'clean_source_max_cosine':float((qs@n.T).max()),'adversarial_source_max_cosine':float((qa@n.T).max())}
    return adversarial.cpu(),{'history':history,'xai':xai_log,'surrogate_proxies':scores,
        'optimizer':'MPC negative-CE ascent, momentum0.9, delta projection in[0,255] units',
        'initial_delta_std_255':16/255}

## Generated-token attention and model lifecycle

Qwen3-VL-8B, layer29, generates a caption with the exact SAGA instruction and50-token
generation budget. Exact SAGA helpers reconstruct the unfiltered generation matrix,
preserving causal row alignment. The approved extension keeps only SAGA content tokens
overlapping complete decoded words matching FINER concept content words under SAGA's
normalization/stop-word rule. Thus`a dog` matches`a brown dog`, but`car` does not match
`carpet`. No synonym matching or lemmatization is used. Each refresh records the generated
caption, matched concepts, character spans and generated-token offsets. No match raises
an error, writes `xai_failure_<step>.json`, and stops the sample without fallback.

Grad-ECLIP continues to use the author n=1 forward with the FINER object phrases.
FINER retains all deduplicated object phrases from the bank; there is no frequency top-3.

The bank VLM is released before attack models load. For Qwen attention, load CLIPs first,
then use available memory to shard the8B model. BF16 is used where supported, otherwise
FP16 as a hardware adaptation; no quantization. This execution adaptation is not a claim
of identical numerical results across GPUs. Finished samples are hash-checked on resume;
partial samples restart from their seed. Old protocol caches have separate identities.


User-confirmed Grad-ECLIP aggregation: normalize each valid concept map by its sum,
resize to32x32 and renormalize, then take the equal-weight mean. This is a CARA
aggregation choice; the author Grad-ECLIP functions remain unchanged.

In [ ]:
def run_identity(revisions):
    names=['SEED','K','STEPS','REFRESH_EVERY','RANDOM_FROM_STEP','EPSILON_255','ALPHA_255',
        'TEMPERATURE','POSITIVE_OMEGA','MOMENTUM_DECAY','INPUT_RES','BACKBONES','CLIP_IDS','CLIP_DEVICE_MAP','ATTACK_DEVICE','QWEN_DEVICE','QWEN_ID',
        'IMAGE_WEIGHTS','TEXT_WEIGHTS','CAPTION_TEMPERATURE','CAPTION_TOP_P','CAPTION_MAX_NEW_TOKENS',
        'MAX_BANK_ATTEMPTS','QWEN_ATTENTION_LAYER','QWEN_MAX_PIXELS','QWEN_GPU_RESERVE_GIB','XAI_MAX_SIDE',
        'SAVED_TENSORS_ON_CPU','CAPTION_PROMPT','ATTENTION_PROMPT','XAI_METHOD']
    config={n:globals()[n] for n in names}
    config.update(model_revisions={k:v for k,v in revisions.items() if k!=EVAL_MODEL_ID},
        author_sources=AUTHOR_SHA256, notebook_code_sha256=NOTEBOOK_CODE_SHA256,
        live_code_sha256=live_code_digest(),
        runtime={p:package_metadata.version(p) for p in ('torch','torchvision','transformers','numpy','Pillow')},
        cuda=torch.version.cuda, hardware=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    return config

def live_code_digest():
    # Ignore notebook cell filenames/line numbers, but invalidate caches on live code edits.
    import types, inspect
    def constant(value):
        if isinstance(value,types.CodeType): return describe(value)
        if isinstance(value,(tuple,list)): return [constant(c) for c in value]
        if isinstance(value,frozenset): return sorted([constant(c) for c in value],key=repr)
        if isinstance(value,bytes): return {'bytes':value.hex()}
        return repr(value)
    def describe(code):
        return {'bytecode':code.co_code.hex(),'names':list(code.co_names),
            'argcount':code.co_argcount,'kwonly':code.co_kwonlyargcount,
            'consts':[constant(c) for c in code.co_consts]}
    result={}
    namespace=globals()
    for name in sorted(namespace):
        value=namespace[name]
        if isinstance(value,types.FunctionType) and value.__globals__ is namespace:
            result[name]=describe(inspect.unwrap(value).__code__)
        elif isinstance(value,type) and value.__module__==__name__:
            for key,method in vars(value).items():
                if isinstance(method,types.FunctionType): result[name+'.'+key]=describe(inspect.unwrap(method).__code__)
    return digest(result)

def bank_for_pair(pair,captioner,run_dir,identity):
    folder=run_dir/'banks';folder.mkdir(exist_ok=True)
    path=folder/f'{pair["index"]:05d}.json'
    bank_identity={'pair_index':pair['index'],'inputs':pair['input_sha256'],'run':digest(identity)}
    if path.exists():
        record=json.loads(path.read_text(encoding='utf-8'))
        payload={k:v for k,v in record.items() if k!='record_sha256'}
        if record['identity']!=bank_identity or digest(payload)!=record.get('record_sha256'):
            raise RuntimeError(f'Caption bank identity/hash mismatch: {path}')
        if any(len(record[r]['captions'])!=K for r in ('source','target')): raise ValueError('Invalid cached bank K.')
        return record
    record={'identity':bank_identity}
    for j,role in enumerate(('source','target')):
        seed_everything(SEED+pair['index']*1000+j)
        record[role]=caption_bank(captioner,load_attack_image(pair[role]))
    record['record_sha256']=digest(record)
    atomic_json(path,record)
    return record

def sample_complete(folder,expected):
    path=folder/'complete.json'
    if not path.exists(): return False
    record=json.loads(path.read_text(encoding='utf-8'))
    if record['identity']!=expected: raise RuntimeError(f'Completed sample identity mismatch: {folder}')
    for filename,sha in record['files'].items():
        p=folder/filename
        if not p.is_file() or file_hash(p)!=sha: raise RuntimeError(f'Artifact corrupted/missing: {p}')
    return True

def export_sample(folder,pair,source,target,adv,record,identity):
    delta=(adv-source).abs()
    if not torch.isfinite(adv).all() or float(delta.max())>EPSILON_255/255+1e-6:
        raise AssertionError('Adversarial tensor exceeds pixel constraint.')
    for role,x in [('source',source),('target',target),('adversarial',adv)]:
        tensor_image(x).save(folder/f'{role}.png')
    saved_source=np.asarray(Image.open(folder/'source.png')).astype(np.int16)
    saved_adv=np.asarray(Image.open(folder/'adversarial.png')).astype(np.int16)
    saved_linf=int(np.abs(saved_adv-saved_source).max())
    if saved_linf>EPSILON_255: raise AssertionError('Saved PNG violates epsilon.')
    record.update(index=pair['index'],inputs=pair,identity=identity,
        tensor_linf_255=float(delta.max()*255),saved_linf_255=saved_linf,
        source_size=list(source.shape[-2:]),target_size=list(target.shape[-2:]))
    atomic_json(folder/'attack.json',record)
    # Complete marker written last; partial samples are restarted from their deterministic seed.
    files={str(p.relative_to(folder)).replace('\\','/'):file_hash(p)
           for p in folder.rglob('*') if p.is_file() and p.name!='complete.json'}
    atomic_json(folder/'complete.json',{'identity':identity,'files':files})

def export_manifest(run_dir,pairs):
    rows=[]
    for pair in pairs:
        folder=run_dir/'samples'/f'{pair["index"]:05d}'
        if not (folder/'complete.json').exists(): continue
        row={'index':pair['index'],'sample_id':f'{pair["index"]:05d}',
            'images':{r:f'samples/{pair["index"]:05d}/{r}.png' for r in ('source','target','adversarial')},
            'original_paths':{r:pair[r] for r in ('source','target')},
            'caption_bank':f'banks/{pair["index"]:05d}.json'}
        rows.append(row)
    (run_dir/f'manifest_{RUN_MODE}_{SAMPLE_START}.jsonl').write_text(
        ''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
    return rows

## Đánh giá sau attack

Hai mức đánh giá được giữ riêng:

- **Kiểm tra kỹ thuật:** đủ K/K caption, nearest-mean index, loss/gradient hữu hạn, đúng schedule,
  map đủ 11 mốc, pixel epsilon trên tensor **và PNG**, kích thước source giữ nguyên.
- **Surrogate proxies:** global InfoNCE trước/sau, cosine target/source, khoảng cách caption
  của VLM đánh giá tới banks. Các chỉ số này không phải ground-truth targeted ASR.

VLM đánh giá mặc định Qwen3-VL, khác model sinh bank/XAI nhưng vẫn cùng họ Qwen.
VLM chỉ caption ảnh clean/adv; **không nhận target caption** để tránh gợi ý target trong prompt.
Target/source reference vẫn là bank đóng băng. CLIP tính text similarity trên caption của VLM đánh giá
là proxy thuộc surrogate; không khẳng định đây là đánh giá độc lập hoàn toàn.

File `human_review_<mode>_<start>_<evaluator_id>.csv` để reviewer điền `target_success` và
`clean_already_target` (0/1). Báo cáo ASR chỉ xuất khi **tất cả** mẫu được gán nhãn;
báo cả tỷ lệ trên toàn bộ N và tỷ lệ có điều kiện trên clean chưa giống target. Nhãn không được
tự sinh bằng threshold CLIP. Đối chiếu caption/ảnh trực tiếp ở preview trước khi chạy full.
`target_success=1` khi **caption adversarial do VLM đánh giá sinh ra** mô tả đúng chủ thể/object
đích theo target image và bank; `clean_already_target=1` khi caption clean đã đáp ứng cùng tiêu chí.
Không gán thành công chỉ vì caption khác source hoặc chứa một từ trùng target. Ghi trường hợp
mơ hồ vào `review_notes` và thống nhất rubric trước full run. Mỗi evaluator protocol có bảng riêng;
đổi evaluator không reuse nhãn của evaluator cũ. Nếu tắt caption evaluation, chỉ xuất bảng ảnh
để kiểm tra kỹ thuật, không báo VLM ASR.

In [ ]:
def evaluate_caption_outputs(run_dir,rows,revisions):
    eval_config={'model':EVAL_MODEL_ID,'revision':revisions[EVAL_MODEL_ID],
        'prompt':EVAL_CAPTION_PROMPT,'max_new_tokens':CAPTION_MAX_NEW_TOKENS,
        'do_sample':False,'max_pixels':QWEN_MAX_PIXELS,'transformers':package_metadata.version('transformers')}
    eval_dir=run_dir/'evaluation'/digest(eval_config)[:16];eval_dir.mkdir(parents=True,exist_ok=True)
    atomic_json(eval_dir/'protocol.json',eval_config)
    evaluator=None;outputs=[]
    try:
        for row in rows:
            path=eval_dir/f'{row["index"]:05d}.json'
            inputs={r:file_hash(run_dir/row['images'][r]) for r in ('source','adversarial')}
            expected={'inputs':inputs,'protocol':digest(eval_config)}
            if path.exists():
                record=json.loads(path.read_text(encoding='utf-8'))
                if record['identity']!=expected: raise RuntimeError('Evaluation cache identity mismatch.')
                payload={k:v for k,v in record.items() if k!='record_sha256'}
                if record.get('record_sha256')!=digest(payload): raise RuntimeError('Evaluation response cache corrupted.')
            else:
                if evaluator is None: evaluator=QwenCaptioner(EVAL_MODEL_ID,revisions[EVAL_MODEL_ID],EVAL_DEVICE)
                seed_everything(SEED+row['index'])
                captions={r:evaluator.caption(open_rgb(run_dir/row['images'][r]),evaluation=True)
                          for r in ('source','adversarial')}
                record={'identity':expected,'index':row['index'],'captions':captions}
                record['record_sha256']=digest(record);atomic_json(path,record)
            outputs.append(record)
    finally:
        del evaluator;free_cuda()
    # Staged lifecycle: evaluate VLM first, release it, then load one CLIP at a time for text proxies.
    from transformers import CLIPModel,AutoTokenizer
    scores={r['index']:{} for r in rows}
    lookup={o['index']:o for o in outputs}
    for name in BACKBONES:
        ident=CLIP_IDS[name]
        model=CLIPModel.from_pretrained(ident,revision=revisions[ident],torch_dtype=torch.float32)
        model.eval().requires_grad_(False).to(EVAL_DEVICE)
        tokenizer=AutoTokenizer.from_pretrained(ident,revision=revisions[ident])
        with torch.no_grad():
            for row in rows:
                bank=json.loads((run_dir/row['caption_bank']).read_text(encoding='utf-8'))
                caps=lookup[row['index']]['captions']
                texts=[caps['source'],caps['adversarial']]+bank['target']['captions']+bank['source']['captions']
                tokens=tokenizer(texts,padding=True,return_tensors='pt',truncation=False).to(EVAL_DEVICE)
                if tokens.input_ids.shape[1]>model.config.text_config.max_position_embeddings:
                    raise ValueError('Evaluation caption exceeds CLIP token budget; inspect raw caption.')
                emb=unit(model.get_text_features(**tokens))
                tgt=unit(emb[2:2+K].mean(0,keepdim=True));src=unit(emb[2+K:].mean(0,keepdim=True))
                similarities=emb[:2]@torch.cat([tgt,src]).T
                scores[row['index']][name]={'clean_target':float(similarities[0,0]),
                    'adv_target':float(similarities[1,0]),'clean_source':float(similarities[0,1]),
                    'adv_source':float(similarities[1,1])}
        del model,tokenizer;free_cuda()
    results=[]
    for row in rows:
        idx=row['index'];record={'index':idx,'evaluation_protocol':digest(eval_config),**lookup[idx]['captions']}
        for key in ('clean_target','adv_target','clean_source','adv_source'):
            record[key+'_text_cosine_proxy']=float(np.mean([scores[idx][m][key] for m in BACKBONES]))
        record['target_text_cosine_gain_proxy']=record['adv_target_text_cosine_proxy']-record['clean_target_text_cosine_proxy']
        results.append(record)
    pd.DataFrame(results).to_csv(eval_dir/f'caption_proxies_{RUN_MODE}_{SAMPLE_START}.csv',index=False)
    atomic_json(eval_dir/f'raw_scores_{RUN_MODE}_{SAMPLE_START}.json',scores)
    return eval_dir,results

def human_review_table(run_dir,rows,caption_records):
    protocols={r['evaluation_protocol'] for r in caption_records}
    if len(protocols)>1: raise ValueError('Mixed evaluation protocols.')
    protocol=next(iter(protocols)) if protocols else 'image_only_no_vlm'
    path=run_dir/f'human_review_{RUN_MODE}_{SAMPLE_START}_{protocol[:16]}.csv'
    if path.exists():
        frame=pd.read_csv(path)
        if frame['index'].tolist()!=[r['index'] for r in rows]: raise ValueError('Review table indices mismatch.')
        return path
    lookup={r['index']:r for r in caption_records}
    records=[]
    for row in rows:
        bank=json.loads((run_dir/row['caption_bank']).read_text(encoding='utf-8'))
        record={'index':row['index'],'evaluation_protocol':protocol,**{r:str(run_dir/p) for r,p in row['images'].items()},
            'source_bank':json.dumps(bank['source']['captions'],ensure_ascii=False),
            'target_bank':json.dumps(bank['target']['captions'],ensure_ascii=False),
            'clean_vlm_caption':lookup.get(row['index'],{}).get('source',''),
            'adv_vlm_caption':lookup.get(row['index'],{}).get('adversarial',''),
            'target_success':'','clean_already_target':'','review_notes':''}
        records.append(record)
    pd.DataFrame(records).to_csv(path,index=False)
    return path

def report_human_asr(review_csv):
    frame=pd.read_csv(review_csv)
    if 'evaluation_protocol' in frame and (frame['evaluation_protocol']=='image_only_no_vlm').any():
        return {'status':'no_vlm_caption_evaluation','n':len(frame)}
    for col in ('target_success','clean_already_target'):
        if frame[col].isna().any():
            return {'status':'pending_human_labels','n':len(frame),'labeled':int(frame['target_success'].notna().sum())}
        if not frame[col].isin([0,1]).all(): raise ValueError(f'{col} must contain only 0/1.')
    eligible=frame[frame['clean_already_target']==0]
    return {'status':'human_labeled','n':len(frame),'targeted_asr_all':float(frame['target_success'].mean()),
        'n_clean_not_target':len(eligible),
        'targeted_asr_clean_not_target':float(eligible['target_success'].mean()) if len(eligible) else None}

In [ ]:
def self_checks():
    verify_author_sources();torch.manual_seed(17)
    q=torch.randn(1,9,requires_grad=True);p=torch.randn(1,9);n=torch.randn(1,9)
    actual=mpc_infonce_score(q,p,n,.2,2.)
    expected=MPC_NS['info_nce_loss'](q,p,n,temperature=.2,omega=2.)
    torch.testing.assert_close(actual,expected)
    g,=torch.autograd.grad(actual,q)
    assert mpc_infonce_score(q+.001*g,p,n,.2,2.)>actual
    assert crop_area(1)==.5 and crop_area(150)==13/18 and crop_area(300)==1.
    assert crop_mode(149)=='xai' and crop_mode(150)=='random'
    assert sample_count()==NUM_SAMPLES
    return {'status':'passed','checks':['author_sha256','K1_equals_author_MPC',
        'MPC_ascent_direction','crop_0.5_to_1','random_from150']}
SELF_CHECK_REPORT=self_checks()
print(SELF_CHECK_REPORT)

In [ ]:
# Resolve and display pair ordering before pretrained inference. No changes to image dimensions.
PAIRS=[];RUN_DIR=None;MANIFEST_ROWS=[];CAPTION_EVALUATION=[]
if RUN_EXPERIMENT:
    validate_config()
    PAIRS=prepare_pairs()
    from IPython.display import display
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(min(5,len(PAIRS)),2,figsize=(10,3*min(5,len(PAIRS))),squeeze=False)
    for row,axes_row in zip(PAIRS,axes):
        for role,ax in zip(('source','target'),axes_row):
            image=load_attack_image(row[role]);ax.imshow(image);ax.axis('off')
            ax.set_title(f'{row["index"]} {role}: {image.size}')
    plt.tight_layout();plt.show()
    display(pd.DataFrame(PAIRS)[['index','source','target']])

In [ ]:
def execute_experiment(pairs):
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    revisions=resolve_revisions();identity=run_identity(revisions)
    run_dir=OUTPUT_ROOT/(XAI_METHOD+'_'+digest(identity)[:16]);run_dir.mkdir(parents=True,exist_ok=True)
    atomic_json(run_dir/'config.json',identity)
    atomic_json(run_dir/'self_checks.json',SELF_CHECK_REPORT)
    print('Run directory:',run_dir)
    captioner=None;ensemble=None;xai=None
    try:
        # Stage 1: bank creation. Reuse banks only if each identity/hash matches.
        if any(not (run_dir/'banks'/f'{p["index"]:05d}.json').exists() for p in pairs):
            captioner=QwenCaptioner(QWEN_ID,revisions[QWEN_ID],QWEN_DEVICE,
                                    attention=(XAI_METHOD=='qwen_attention'))
        banks={p['index']:bank_for_pair(p,captioner,run_dir,identity) for p in pairs}
        # Release bank model before placing CLIPs / the8B attention model.
        del captioner;captioner=None;free_cuda()
        pending=[]
        for pair in pairs:
            folder=run_dir/'samples'/f'{pair["index"]:05d}';folder.mkdir(parents=True,exist_ok=True)
            expected={'run':digest(identity),'inputs':pair['input_sha256'],
                'bank':banks[pair['index']]['record_sha256'],'index':pair['index']}
            if not sample_complete(folder,expected): pending.append((pair,folder,expected))
        if pending:
            ensemble=ClipEnsemble(revisions)
            if XAI_METHOD=='qwen_attention':
                if captioner is None:
                    captioner=QwenCaptioner(QWEN_ID,revisions[QWEN_ID],QWEN_DEVICE,attention=True)
                xai=QwenAttentionXAI(captioner)
            else: xai=GradEclipXAI(QWEN_DEVICE)
            for pair,folder,expected in pending:
                print('Attacking',pair['index'],XAI_METHOD)
                src=image_tensor(load_attack_image(pair['source']));tgt=image_tensor(load_attack_image(pair['target']))
                adv,record=attack_one(ensemble,xai,src,tgt,banks[pair['index']],folder,SEED+pair['index'])
                export_sample(folder,pair,src,tgt,adv,record,expected)
                export_manifest(run_dir,pairs)  # retain a usable partial manifest after interruption
                del adv,src,tgt;free_cuda()
    finally:
        del ensemble,xai,captioner;free_cuda()
    rows=export_manifest(run_dir,pairs)
    if len(rows)!=len(pairs): raise RuntimeError('Incomplete attack manifest.')
    evaluation=[]
    if EVALUATE_CAPTIONS:
        eval_dir,evaluation=evaluate_caption_outputs(run_dir,rows,revisions)
        print('Caption evaluation:',eval_dir)
    review=human_review_table(run_dir,rows,evaluation)
    atomic_json(review.with_suffix('.asr.json'),report_human_asr(review))
    print('Review table:',review)
    return run_dir,rows,evaluation

if RUN_EXPERIMENT:
    RUN_DIR,MANIFEST_ROWS,CAPTION_EVALUATION=execute_experiment(PAIRS)

In [ ]:
# Artifact checks and review; also useful after an interrupted session by setting RUN_DIR manually.
if RUN_DIR is not None:
    import matplotlib.pyplot as plt
    from IPython.display import display
    summary=[]
    for row in MANIFEST_ROWS:
        folder=RUN_DIR/'samples'/f'{row["index"]:05d}'
        attack=json.loads((folder/'attack.json').read_text(encoding='utf-8'))
        bank=json.loads((RUN_DIR/row['caption_bank']).read_text(encoding='utf-8'))
        assert [r['step'] for r in attack['xai']]==list(range(0,301,30))
        assert [r['crop_mode'] for r in attack['history']]==[crop_mode(i) for i in range(1,301)]
        assert attack['history'][-1]['box_yxhw']==[0,0,*attack['source_size']]
        assert attack['saved_linf_255']<=EPSILON_255
        summary.append({'index':row['index'],'saved_linf_255':attack['saved_linf_255'],
                        **attack['surrogate_proxies']})
    display(pd.DataFrame(summary))
    if CAPTION_EVALUATION: display(pd.DataFrame(CAPTION_EVALUATION))
    if MANIFEST_ROWS:
        row=MANIFEST_ROWS[0];folder=RUN_DIR/'samples'/f'{row["index"]:05d}'
        bank=json.loads((RUN_DIR/row['caption_bank']).read_text(encoding='utf-8'))
        anchors=json.loads((folder/'anchors.json').read_text(encoding='utf-8'))
        print('Source captions:',bank['source']['captions'])
        print('Target captions:',bank['target']['captions'])
        print('Selected target index/caption:',anchors['target_index'],anchors['target_caption'])
        print('Source concepts used for XAI:',bank['source']['selected_concepts'])
        fig,axes=plt.subplots(1,4,figsize=(16,4))
        for ax,role in zip(axes[:3],('source','target','adversarial')):
            ax.imshow(open_rgb(folder/f'{role}.png'));ax.set_title(role);ax.axis('off')
        s=np.asarray(open_rgb(folder/'source.png')).astype(float)
        a=np.asarray(open_rgb(folder/'adversarial.png')).astype(float)
        axes[3].imshow(np.abs(a-s).max(-1),vmin=0,vmax=EPSILON_255,cmap='magma')
        axes[3].set_title('max |delta| per pixel (0..epsilon)');axes[3].axis('off')
        plt.tight_layout();plt.show()
        attack=json.loads((folder/'attack.json').read_text(encoding='utf-8'))
        fig,axes=plt.subplots(1,4,figsize=(16,4))
        for ax,step in zip(axes[:3],(0,150,300)):
            heat=np.load(folder/'xai'/f'xai_{step:03d}.npz')['aggregate_32']
            ax.imshow(heat,cmap='inferno');ax.set_title(f'XAI step {step}');ax.axis('off')
        axes[3].plot([r['step'] for r in attack['xai']],
                     [r['dispersion']['normalized_entropy'] for r in attack['xai']],marker='o')
        axes[3].axvline(150,color='gray',linestyle='--');axes[3].set_ylim(0,1.05)
        axes[3].set(xlabel='completed updates',ylabel='normalized XAI entropy')
        plt.tight_layout();plt.show()
    print(f'Processed {len(MANIFEST_ROWS)}/{NUM_SAMPLES} pairs. To scale: set NUM_SAMPLES=200 or 1000; RUN_MODE="full" labels artifacts. Then Run all.')